In [1]:
import numpy as np
from scipy.linalg import expm

def verify_dpi_capacity_degradation(channel_N, correction_C, dim=2):
    """
    Verify that C_χ(C∘N) ≤ C_χ(N) for given channels.
    """
    # Function to compute Holevo capacity (simplified version)
    def holevo_capacity(channel, num_states=4, max_iter=1000):
        # Use optimization or analytical bounds
        # For demonstration, we'll use a simplified approach
        best_chi = 0
        for _ in range(num_states):
            # Generate random ensemble
            probs = np.random.dirichlet(np.ones(num_states))
            states = [random_density_matrix(dim) for _ in range(num_states)]
            
            # Compute Holevo quantity
            channel_states = [channel(state) for state in states]
            rho_avg = sum(p*rho for p, rho in zip(probs, channel_states))
            
            S_avg = von_neumann_entropy(rho_avg)
            S_indiv = sum(p*von_neumann_entropy(rho) 
                         for p, rho in zip(probs, channel_states))
            
            chi = S_avg - S_indiv
            best_chi = max(best_chi, chi)
        
        return best_chi
    
    def random_density_matrix(d):
        """Generate random density matrix."""
        A = np.random.randn(d, d) + 1j*np.random.randn(d, d)
        rho = A @ A.conj().T
        return rho / np.trace(rho)
    
    def von_neumann_entropy(rho):
        """Compute von Neumann entropy."""
        eigvals = np.linalg.eigvalsh(rho)
        eigvals = eigvals[eigvals > 1e-12]
        return -np.sum(eigvals * np.log2(eigvals))
    
    # Compute capacities
    C_N = holevo_capacity(channel_N)
    C_CN = holevo_capacity(lambda rho: correction_C(channel_N(rho)))
    
    # Verify DPI
    violation = C_CN - C_N
    degradation = max(0, C_N - C_CN) / C_N if C_N > 0 else 0
    
    return C_N, C_CN, violation, degradation

# Example: Depolarizing channel with amplitude damping correction
def depolarizing_channel(rho, p=0.1):
    """Depolarizing channel."""
    d = rho.shape[0]
    return (1-p)*rho + p*np.eye(d)/d

def amplitude_damping_correction(rho, alpha=0.1):
    """Simple amplitude damping 'correction' (actually adds more noise)."""
    # This is NOT a true correction but demonstrates the principle
    K0 = np.array([[1, 0], [0, np.sqrt(1-alpha)]])
    K1 = np.array([[0, np.sqrt(alpha)], [0, 0]])
    return K0 @ rho @ K0.conj().T + K1 @ rho @ K1.conj().T

# Test
C_N, C_CN, violation, degradation = verify_dpi_capacity_degradation(
    lambda rho: depolarizing_channel(rho, p=0.2),
    lambda rho: amplitude_damping_correction(rho, alpha=0.1)
)

print(f"C_χ(N) = {C_N:.6f}")
print(f"C_χ(C∘N) = {C_CN:.6f}")
print(f"Violation of DPI: {violation:.6f} (should be ≤ 0)")
print(f"Degradation: {degradation*100:.2f}%")

C_χ(N) = 0.274309
C_χ(C∘N) = 0.267873
Violation of DPI: -0.006436 (should be ≤ 0)
Degradation: 2.35%
